# 23 – Logging & Observability

Tests the structured logging utilities used across all agents and graph nodes:
- `get_logger` — returns a named logger with structlog formatting
- `log_agent_call` — wraps agent invocations with timing + metadata
- `log_graph_event` — structured events for graph node transitions

Also covers how logs flow during a real pipeline run.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
os.environ['ENABLE_MOCK'] = 'true'

## 1. get_logger

In [ ]:
from core.logging_utils import get_logger

logger = get_logger('notebooks.test')
print('Logger name:', logger.name)

logger.info('Test info message', extra={'query': 'test', 'agent': 'information_agent'})
logger.warning('Test warning', extra={'anomaly_count': 2})
logger.debug('Debug message — visible at DEBUG level')

## 2. log_agent_call — decorator

In [ ]:
from core.logging_utils import log_agent_call
from core.base_agent import AgentRequest, AgentResult

@log_agent_call(agent_name='echo_agent')
def execute(request: AgentRequest) -> AgentResult:
    return AgentResult(success=True, message=f'Echo: {request.query}', confidence=1.0)

req = AgentRequest(query='test logging')
result = execute(req)
print('Result:', result.message)

## 3. log_graph_event — pipeline events

In [ ]:
from core.logging_utils import log_graph_event

log_graph_event('supervisor_node', event='intent_classified', intent='data_quality', confidence=0.91)
log_graph_event('information_node', event='agent_executed', products=['retention'], anomalies=2)
log_graph_event('auto_ticket_node', event='hitl_pending', count=2, thread_id='test-thread')
log_graph_event('synthesizer_node', event='summary_generated', tokens=312)
print('Events logged (check stdout above)')

## 4. Agent-level logging during execution

In [ ]:
import logging

# Set to DEBUG to see full agent call trace
logging.getLogger('agents').setLevel(logging.DEBUG)
logging.getLogger('graph').setLevel(logging.DEBUG)

from agents.information_agent import InformationAgent
from core.base_agent import AgentRequest

agent = InformationAgent()
result = agent.execute(AgentRequest(query='retention metrics', data_products=['retention']))

print('Agent execution complete')
print('Success:', result.success)
print('Exec ms:', result.execution_time_ms)

## 5. Error logging — exception capture

In [ ]:
from core.logging_utils import get_logger

logger = get_logger('test.errors')

try:
    raise ConnectionError('Databricks host unreachable: adb-xxxxx.azuredatabricks.net')
except Exception as exc:
    logger.error('Agent execution failed', exc_info=True, extra={
        'agent': 'information_agent',
        'query': 'retention metrics',
        'retry_attempt': 3,
    })

print('Exception logged (see traceback in output above)')

## 6. Log levels per module

In [ ]:
import logging

modules = [
    'api.app',
    'agents',
    'graph',
    'core',
    'services',
    'teams',
    'mcp_server',
]

print('Current log levels:')
for mod in modules:
    lvl = logging.getLogger(mod).level
    lvl_name = logging.getLevelName(lvl) if lvl != 0 else 'NOTSET (inherits root)'
    print(f'  {mod:<25} {lvl_name}')

root = logging.getLogger()
print(f'\n  root                     {logging.getLevelName(root.level)}')